# EDA: Bank Customer Churn Dataset
## Análisis Exhaustivo para Modelamiento Predictivo Avanzado

### Imports y Configuración

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('Imports OK')

## 1. CARGA Y EXPLORACIÓN INICIAL

In [ ]:
with open('customer_churn_data.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f'Shape: {df.shape}')
print(f'\nPrimeras filas:')
print(df.head())
print(f'\nInfo:')
print(df.info())

## 2. ANÁLISIS DE NULOS (MISSINGNESS)

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

print('Features con >50% nulos:')
print(missing_pct[missing_pct > 50])
print(f'\nTotal features con >75% nulos: {(missing_pct > 75).sum()}')
print(f'Total features con >80% nulos: {(missing_pct > 80).sum()}')

In [ ]:
# Visualizar missingness
plt.figure(figsize=(12, 6))
missing_top = missing_pct[missing_pct > 0].head(25)
plt.barh(range(len(missing_top)), missing_top.values, color='coral')
plt.yticks(range(len(missing_top)), missing_top.index, fontsize=8)
plt.xlabel('% de nulos')
plt.title('Top 25 Features con Valores Faltantes')
plt.tight_layout()
plt.show()

## 3. DISTRIBUCIÓN DEL TARGET

In [ ]:
print('Distribución de churn_90d:')
print(df['churn_90d'].value_counts())
print('\nProporción:')
print(df['churn_90d'].value_counts(normalize=True))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

df['churn_90d'].value_counts().plot(kind='bar', ax=ax1, color=['#2ecc71', '#e74c3c'])
ax1.set_title('Conteo de Clientes')
ax1.set_xticklabels(['No Churn', 'Churn'], rotation=0)
ax1.set_ylabel('Cantidad')

df['churn_90d'].value_counts().plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
ax2.set_ylabel('')
ax2.set_title('Proporción')

plt.tight_layout()
plt.show()

## 4. DISTRIBUCIONES Y VARIANZA DE FEATURES

In [ ]:
numeric_features = df.select_dtypes(include=[np.number]).drop('churn_90d', axis=1).columns.tolist()

print(f'Total features numéricas: {len(numeric_features)}')
print(f'\nEstadísticas descriptivas:')
print(df[numeric_features].describe())

In [ ]:
# Varianza de features
variance = df[numeric_features].var().sort_values()

print(f'Features con varianza < 1: {(variance < 1).sum()}')
print(f'Features con varianza < 0.1: {(variance < 0.1).sum()}')
print('\nTop 20 features con MENOR varianza:')
print(variance.head(20))

In [ ]:
# Histogramas de muestra
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
axes = axes.flatten()

np.random.seed(42)
sample_features = np.random.choice(numeric_features, 12, replace=False)

for i, feat in enumerate(sample_features):
    axes[i].hist(df[feat].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{feat}\n(μ={df[feat].mean():.2f})', fontsize=8)
    axes[i].set_ylabel('Frecuencia', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para detectar outliers
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
axes = axes.flatten()

for i, feat in enumerate(sample_features):
    axes[i].boxplot(df[feat].dropna())
    axes[i].set_title(f'{feat}', fontsize=8)
    axes[i].set_ylabel('Valor', fontsize=8)

plt.tight_layout()
plt.show()

print('Busca features con bigotes largos (outliers extremos)')

## 5. ANÁLISIS DE CORRELACIONES

In [ ]:
corr_with_churn = df[numeric_features + ['churn_90d']].corr()['churn_90d'].drop('churn_90d').abs().sort_values(ascending=False)

print('Top 20 features MÁS CORRELACIONADAS con churn:')
print(corr_with_churn.head(20))
print(f'\nFeatures con |corr| < 0.01 (ruido puro): {(corr_with_churn < 0.01).sum()}')
print(f'Features con |corr| < 0.05: {(corr_with_churn < 0.05).sum()}')

In [ ]:
# Visualizar correlaciones con churn
plt.figure(figsize=(10, 8))
corr_with_churn.head(25).plot(kind='barh', color='steelblue')
plt.xlabel('Correlación Absoluta con Churn')
plt.title('Top 25 Features por Correlación con Churn')
plt.tight_layout()
plt.show()

In [ ]:
# Colinealidad: features casi idénticas
corr_matrix = df[numeric_features].corr()

high_corr_pairs = 0
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.95:
            high_corr_pairs += 1

print(f'Pares de features con |correlación| > 0.95: {high_corr_pairs}')
print('\nEsto indica COLINEALIDAD EXTREMA (features redundantes)')

In [ ]:
# Heatmap de correlaciones (muestra)
plt.figure(figsize=(14, 10))
np.random.seed(42)
sample_for_heatmap = np.random.choice(numeric_features, 20, replace=False)
sns.heatmap(df[sample_for_heatmap].corr(), cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación (muestra de 20 features)')
plt.tight_layout()
plt.show()

## 6. OUTLIERS EXTREMOS

In [ ]:
extreme_features = []
for feat in numeric_features:
    max_val = df[feat].max()
    if max_val > 1e6:
        extreme_features.append((feat, max_val))

extreme_features.sort(key=lambda x: x[1], reverse=True)

print(f'Features con valores máx > 1e6 (outliers extremos):')
for feat, val in extreme_features[:15]:
    print(f'  {feat}: {val:.2e}')

print(f'\nTotal features con outliers extremos: {len(extreme_features)}')

## 7. RESUMEN DIAGNÓSTICO FINAL

In [ ]:
print('='*70)
print('DIAGNÓSTICO FINAL DEL DATASET')
print('='*70)
print(f'''
PROBLEMAS IDENTIFICADOS:

1. RUIDO PURO
   Features sin poder predictivo (|corr| < 0.01): {(corr_with_churn < 0.01).sum()}
   -> No contienen información sobre churn

2. NULOS MASIVOS
   Features con >75% nulos: {(missing_pct > 75).sum()}
   -> Prácticamente vacíos, sin valor

3. BAJA VARIANZA
   Features con varianza < 1: {(variance < 1).sum()}
   -> Sin poder discriminativo

4. COLINEALIDAD
   Pares con |corr| > 0.95: {high_corr_pairs}
   -> Redundancia extrema

5. OUTLIERS EXTREMOS
   Features con valores > 1e6: {len(extreme_features)}
   -> Distorsionan el escalado

6. DESBALANCE DE CLASES
   No-Churn: {(df['churn_90d']==0).sum()} ({(df['churn_90d']==0).sum()/len(df)*100:.1f}%)
   Churn: {(df['churn_90d']==1).sum()} ({(df['churn_90d']==1).sum()/len(df)*100:.1f}%)

FEATURES CON SEÑAL REAL (|corr| > 0.05):
{list(corr_with_churn[corr_with_churn > 0.05].index)}

RECOMENDACIONES PARA PARTE 2:
1. Eliminar features con >80% nulos
2. Eliminar features con varianza < 1
3. Eliminar features ruido puro (|corr| < 0.01)
4. Eliminar features colineales (mantener 1 por par)
5. Usar RobustScaler para manejar outliers extremos
6. Usar GradientBoosting con regularización fuerte
7. Considerar SMOTE o class_weight para desbalance
''')
print('='*70)